# Laboratorium 1 - powtórka z kryptografii klucza publicznego

## Powtórka z algebry
#### Potęgowanie w grupie

Niech $(G,\circ)$ będzie grupą z działaniem $\circ$ i elementem neutralnym $1_G$. Wtedy dla dowolnego elementu $a\in G$ i $k\in\mathbb{Z}$ definiujemy *potęgę* $$a^k =\left\{\begin{array}{cc}
\underbrace{a\circ a\circ \ldots \circ a}_{k}&\text{ dla }k>0\\
1_G&\text{ dla }k=0\\
\underbrace{a^{-1}\circ a^{-1}\circ \ldots \circ a^{-1}}_{|k|}&\text{ dla }k<0
\end{array}\right.$$
gdzie $a^{-1}$ jest elementem odwrotnym do $a$.

#### Logarytm dyskretny w grupie

Niech $(G,\circ)$ będzie grupą z działaniem $\circ$ i elementem neutralnym $1_G$. Dla $a,b\in G$, $b\neq 1_G$, *logarytmem dyskretnym* $\log_b a$ jest każda liczba $k\in\mathbb{Z}$ taka, że $b^k=a$ (o ile w ogóle taka liczba istnieje).

#### Grupy cykliczne

Niech $(G,\circ)$ będzie grupą z działaniem $\circ$ i elementem neutralnym $1_G$. Grupę $G$ nazywamy *grupą cykliczną*, jeżeli istnieje element $g\in G$ taki, że $G=\{g^k:k\in\mathbb{Z}\}$. Element $g$ nazywamy *generatorem grupy* $G$.

#### Logarytm dyskretny w grupie cyklicznej

Niech $(G,\circ)$ będzie grupą **cykliczną** z działaniem $\circ$, elementem neutralnym $1_G$ i generatorem $g\in G$. Wtedy dla dowolnego $a\in G$ istnieje $k\in\mathbb{Z}$ takie, że $k=\log_g a$.


## Algorytm wymiany klucza Diffiego-Hellmana w grupie cyklicznej

Alice i Bob uzgadniają klucz publiczny $pk$ będący grupą cykliczną $(G,\circ)$ oraz jej generatorem $g$.
- sekret Alice $s_A$: liczba całkowita $n\in \mathbb{Z}\setminus\{0\}$
- sekret Boba $s_B$: liczba całkowita $m\in \mathbb{Z}\setminus\{0\}$
- Alice generuje $x=g^n$ i wysyła do Boba
- Bob generuje $y=g^m$ i wysyła Alice
- Alice oblicza klucz sesji $K=y^n$
- Bob oblicza klucz sesji $K=x^m$

Dlaczego otrzymany klucz $K$ jest taki sam po obydwu stronach?
$$K=y^n=(g^m)^n=g^{mn}$$
$$K=x^m=(g^n)^m=g^{nm}=g^{nm}$$


### Zadanie 1.

Zaimplementuj algorytm wymiany klucza Diffiego-Hellmana w grupie cyklicznej $\mathbb{Z}_{1009}$ z $g=11$. Sprawdź w testach, czy otrzymany klucz sesji faktycznie jest równy po obydwu stronach.

In [39]:
from io import BufferedIOBase
import random

class Person:
    def __init__(self, g: int, Z: int):
        self.g = g
        self.Z = Z
        self.secret = random.randrange(1, Z)   # n | m
        self.public_msg = None                 # x | y
        self.session_key = None


    def generate_public_msg(self, ) -> int:
        self.public_msg = pow(self.g, self.secret, self.Z)
        return self.public_msg

    def get_session_key(self, other_side_public_msg: int):
        self.session_key = pow(other_side_public_msg, self.secret, self.Z)
        return self.session_key

for x in range(100):

    bob = Person(g=11, Z=1009)
    alice = Person(g=11, Z=1009)

    bob_pub = bob.generate_public_msg()
    alice_pub = alice.generate_public_msg()

    print(f"Bob   pub  {bob_pub}")
    print(f"Alice pub  {alice_pub}\n")

    bob_session_key = bob.get_session_key(alice_pub)
    alice_session_key = alice.get_session_key(bob_pub)

    print(f"Bob   session_key  {bob_session_key}")
    print(f"Alice session_key  {alice_session_key}\n")

    assert bob_session_key == alice_session_key, "klucze sesji powinny być równe"

Bob   pub  718
Alice pub  881

Bob   session_key  432
Alice session_key  432

Bob   pub  643
Alice pub  917

Bob   session_key  853
Alice session_key  853

Bob   pub  983
Alice pub  839

Bob   session_key  657
Alice session_key  657

Bob   pub  408
Alice pub  37

Bob   session_key  251
Alice session_key  251

Bob   pub  609
Alice pub  287

Bob   session_key  105
Alice session_key  105

Bob   pub  744
Alice pub  866

Bob   session_key  557
Alice session_key  557

Bob   pub  210
Alice pub  724

Bob   session_key  872
Alice session_key  872

Bob   pub  481
Alice pub  823

Bob   session_key  43
Alice session_key  43

Bob   pub  105
Alice pub  125

Bob   session_key  302
Alice session_key  302

Bob   pub  593
Alice pub  680

Bob   session_key  385
Alice session_key  385

Bob   pub  211
Alice pub  374

Bob   session_key  1
Alice session_key  1

Bob   pub  873
Alice pub  501

Bob   session_key  881
Alice session_key  881

Bob   pub  93
Alice pub  22

Bob   session_key  571
Alice session_key  

## Krzywe eliptyczne nad $\mathbb{R}$

#### Co nazywamy krzywą eliptyczną?
Krzywą daną uproszczonym równaniem Weierstrassa$$y^2=x^3+ax+b$$gdzie $a,b\in\mathbb{R}$ nazywamy *krzywą eliptyczną*, jeżeli wielomian po prawej stronie ma dokładnie trzy różne pierwiastki. Ma to miejsce wtedy i tylko wtedy, gdy jej tzw. *wyróżnik* $\Delta=-16(4a^3+27b^2)$ jest różny od zera.

#### Dodawanie punktów na krzywej eliptycznej
Niech $E=\{(x,y)\in\mathbb{R}^2: y^2=x^3+ax+b\}$ będzie krzywą eliptyczną i niech $P,Q\in E$. Niech $L$ będzie prostą przechodzącą przez punkty $P$ i $Q$, a jeżeli $P=Q$, to przez $L$ rozumiemy prostą styczną do $E$ w $P$. Niech $R$ będzie punktem przecięcia prostej $L$ z krzywą $E$, różnym od $P$ i $Q$ i niech $L'$ będzie prostą równoległą do osi $OY$ przechodzącą przez punkt $R$. Wtedy $L'$ przecina krzywą $E$ w drugim punkcie, symetrycznym do $R$ względem $OX$. Ten punkt definiujemy jako sumę $P\oplus Q$.

Jeżeli punkty $P$ i $Q$ były symetryczne do siebie względem osi $OX$, to nie znajdziemy trzeciego punktu przecięcia (punktu $R$). Dlatego też do krzywej $E$ "dorzucamy" tzw. *wyróżnony punkt w nieskończoności* oznaczany $\mathcal{O}$.
    
![źródło: J.H. Silverman, The Arithmetic of Elliptic Curves, Springer, 2009](group_law.png)

- Jeżeli punkty $P,Q,R\in E$ są współliniowe, to $(P\oplus Q)\oplus R=\mathcal{O}$.
- Dla każdego $P\in E$ mamy $P\oplus\mathcal{O}=P$.
- Dla dowolnych $P,Q\in E$ mamy $P\oplus Q=Q\oplus P$.
- Dla dowolnego $P\in E$ istnieje punkt $P'\in E$ taki, że $P\oplus P'=\mathcal{O}$.
- Dla dowolnych $P,Q,R\in E$ mamy $(P\oplus Q)\oplus R=P\oplus(Q\oplus R)$.

Z powyższych własności wynika, że zbiór $E\cup\{\mathcal{O}\}$ z działaniem $\oplus$ jest grupą abelową. Zatem dla każdego $G\in E$ mamy grupę cykliczną generowaną przez $G$, postaci $\langle G\rangle$.

## Krzywe eliptyczne nad ciałami skończonymi

#### Co to jest krzywa eliptyczna nad ciałem skończonym $\mathbb{F}$?
*Krzywa eliptyczna* $E$ nad ciałem skończonym $\mathbb{F}$ to para $(E,\mathcal{O})$, gdzie $E$ to gładka krzywa algebraiczna rodzaju 1, a $\mathcal{O}$ jest wyróżnionym punktem $\mathcal{O}\in E$, nazywanym *punktem w nieskończoności*.

#### Jaki związek mają krzywe eliptyczne nad ciałem $\mathbb{F}$ z poznanymi krzywymi eliptycznymi nad $\mathbb{R}$?
Niech $(E,\mathcal{O})$ będzie krzywą eliptyczną nad ciałem skończonym $\mathbb{F}$. Wtedy:

- istnieją funkcje $x,y\in \mathbb{F}(E)$ takie, że odwzorowanie $\varphi:E\to \mathbb{P}^2$, $\varphi=[x,y,1]$, jest izomorfizmem $E/\mathbb{F}$ na krzywą daną równaniem Weierstrassa$$C: Y^2+a_1XY+a_3Y=X^3+a_2X^2+a_4X+a_6,\ a_1,...,a_6\in \mathbb{F},$$
takim, że $\varphi(\mathcal{O})=[0,1,0]$, gdzie tzw. *punkt rzutowy* $[x,y,z]$ odpowiada punktowy afinicznemu $(x/z,y/z)$.
- każda krzywa gładka $C$ stopnia 3 dana równaniem Weierstrassa jest krzywą eliptyczną nad $\mathbb{F}$ z wyróznionym punktem $\mathcal{O}=[0,1,0]$.
- równanie Weierstrassa krzywej eliptycznej $E$ jest jednoznaczne z dokładnością do liniowej zmiany zmiennych postaci $X=u^2X'+r,\ Y=u^3Y'+su^2X'+t,$ gdzie $u\in \mathbb{F}^*$, $r,s,t\in \mathbb{F}$. 

#### Dzielniki krzywej eliptycznej
Niech $(E,\mathcal{O})$ będzie krzywą eliptyczną nad ciałem skończonym $\mathbb{F}$. *Dzielnikami* $E$ nazywamy elementy wolnej grupy abelowej $\text{Div}(E)$ generowanej przez punkty krzywej $E$. Dla $P,Q\in E$ dzielniki $(P)$ i $(Q)$ są równoważne wtedy i tylko wtedy, gdy $P=Q$. Ponadto dla każdego dzielnika $D\in\text{Div}(E)$ stopnia 0 (tzn. o sumie współczynników równej 0) istnieje jedyny punkt $P_D\in E$ taki, że $D$ jest równoważny dzielnikowi $(P_D)-(\mathcal{O})$. Otrzymujemy zatem odwzorowanie $\sigma:\text{Div}^0(E)\ni D\mapsto P_D\in E$ takie, że
- odwzorowanie $\sigma$ jest suriekcją
- $\sigma(D_1)=\sigma(D_2)$ wtedy i tylko wtedy, gdy dzielniki $D_1$ i $D_2$ są równoważne

Dzielniki stopnia 0 tworzą w $\text{Div}(E)$ podgrupę oznaczaną przez $\text{Div}^0(E)$. *Grupą Picarda* $\text{Pic}^0(E)$ nazywamy grupę ilorazową grupy $\text{Div}(E)$ przez relację równoważności dzielników, a zatem odwzorowanie $\sigma:\text{Pic}^0(E)\ni[D]\to P_D\in E$ jest bijekcją z grupy Picarda na krzywą eliptyczną $E$ i wprowadza na $E$ strukturę grupy.
 
#### Dodawanie punktów krzywej eliptycznej
Niech $(E,\mathcal{O})$ będzie krzywą eliptyczną nad ciałem skończonym $K$ daną równaniem Weierstrassa $ Y^2+a_1XY+a_3Y=X^3+a_2X^2+a_4X+a_6,$ $a_1,...,a_6\in \mathbb{F}$. Wtedy $P+Q$ jako dodawanie w grupie Picarda pokrywa się z punktem $P\oplus Q$ otrzymanym z konstrukcji dodawania punktów na krzywej eliptycznej nad $\mathbb{R}$, tzn. jeżeli $P=[x_p,y_p,1]$, $Q=[x_q,y_q,1]$ oraz $P + Q=R$, to $R=[x_R,y_R,1]$ gdzie $$x_R=\lambda^2+a_1\lambda -a_2-x_P-x_Q,\qquad y_R=-(\lambda+a_1)x_R-\nu-a_3,$$
gdzie dla $x_P\neq x_Q$ mamy
        
$$\lambda=\frac{y_Q-y_P}{x_Q-x_P},\qquad\nu=\frac{y_Px_Q-y_Qx_P}{x_Q-x_P}$$
        
a dla $x_P= x_Q$ mamy
        
$$\lambda=\frac{3x_p^2+2a_2x_P+a_4-a_1y_P}{2y_P+a_1x_P+a_3},\qquad\nu=\frac{-x_P^3+a_4x_P+2a_6-a_3y_P}{2y_P+a_1x_P+a_3}.$$
  



In [5]:
class EllipticCurve:
    """
    Elliptic curve over Galois field F_p given by short Weierstrass equation:
        y^2 = x^3 + a*x + b (mod p)

    Point at infinity is represented by None.
    """

    def __init__(self, a, b, p):
        self.a = a
        self.b = b
        self.p = p

        # Check non-singularity:
        if (4 * a**3 + 27 * b**2) % p == 0:
            raise ValueError("Singular curve")

    def is_on_curve(self, P):
        if P is None:
            return True

        x, y = P
        return (y * y - (x**3 + self.a * x + self.b)) % self.p == 0

    def inverse_mod(self, x):
        return pow(x, -1, self.p)

    def negate(self, P):
        if P is None:
            return None

        x, y = P
        return (x, (-y) % self.p)

    def add(self, P, Q):
        """
        Point addition P + Q
        """
        if P is None:
            return Q

        if Q is None:
            return P

        x1, y1 = P
        x2, y2 = Q

        # P + (-P) = O
        if x1 == x2 and (y1 + y2) % self.p == 0:
            return None

        # Point doubling
        if P == Q:
            s = (
                (3 * x1 * x1 + self.a)
                * self.inverse_mod(2 * y1)
            ) % self.p

        # General addition
        else:
            s = (
                (y2 - y1)
                * self.inverse_mod(x2 - x1)
            ) % self.p

        x3 = (s * s - x1 - x2) % self.p
        y3 = (s * (x1 - x3) - y1) % self.p

        return (x3, y3)

    def multiply(self, k, P):
        """
        Scalar multiplication k*P equal by definition to group power P**k.
        Double-and-add algorithm.
        """
        if not self.is_on_curve(P):
            raise ValueError("Point is not on curve")

        result = None
        addend = P

        while k:
            if k & 1:
                result = self.add(result, addend)

            addend = self.add(addend, addend)
            k >>= 1

        return result

    def order_multiple(self, n, P):
        """
        Checks whether nP = O.
        """
        return self.multiply(n, P) is None

### Zadanie 2.

Zaimplementuj algorytm wymiany klucza Diffiego-Hellmana w podgrupie cyklicznej, którą na krzywej $E:y^2=x^3+2x+3$ nad $\mathbb{F}_{97}$ generuje punkt $G=(3,6)$.

In [6]:
#TYPE YOUR CODE BELOW

